# GRPO Multi-GPU Training + vLLM Weight Sync

Single-node **multi-GPU** GRPO training on GSM8K, then **NCCL weight sync** to a running vLLM server.

| | |
|---|---|
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` |
| **Architecture** | 3 GPUs in one pod — vLLM on GPU 0, GRPO DDP training on GPUs 1 & 2, localhost NCCL sync |
| **Training** | `torchrun --nproc_per_node=2` for distributed data-parallel GRPO |
| **Weights** | `--load-format auto` (real checkpoint, not dummy) |

## Flow

1. Deploy vLLM with real weights (3 GPUs, vLLM uses GPU 0 only)
2. **Baseline** — 10 GSM8K questions via vLLM API, score accuracy
3. **GRPO train** — TRL `GRPOTrainer` via `torchrun` on GPUs 1 & 2 (DDP, 64 samples)
4. **Weight sync** — fresh process loads checkpoint on GPU 1, NCCL broadcast to vLLM on GPU 0
5. **Post-sync eval** — same 10 questions, compare accuracy + Prometheus metrics

## Install dependencies

In [1]:
!python3 -m pip install -q -U kubeflow kubernetes openai requests transformers datasets trl accelerate vllm

In [2]:
import json
import os
import subprocess
import time

import requests
from kubernetes import client as k8s
from kubernetes import config as k8s_config
from openai import OpenAI

## Configure cluster access and vLLM settings

In [ ]:
# Cluster bootstrap config for notebook-managed runs.
SA_NS_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"

if not os.getenv("NOTEBOOK_NAMESPACE") and os.path.exists(SA_NS_PATH):
    with open(SA_NS_PATH) as f:
        os.environ["NOTEBOOK_NAMESPACE"] = f.read().strip()

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE", "<your-namespace>")
VLLM_NAME = "grpo-vllm-rollout"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
IMAGE = os.getenv(
    "VLLM_IMAGE",
    "image-registry.openshift-image-registry.svc:5000/grpoxtrainer/vllm-async-init-poc:async-init-poc-20260519-150604",
)
API_KEY = os.getenv("VLLM_API_KEY", "dummy-token")
WEIGHT_TRANSFER_BACKEND = os.getenv("VLLM_WEIGHT_TRANSFER_BACKEND", "nccl")
VLLM_GPU_COUNT = int(os.getenv("VLLM_GPU_COUNT", "3"))
VLLM_CPU_REQUEST = os.getenv("VLLM_CPU_REQUEST", "2")
VLLM_MEMORY_REQUEST = os.getenv("VLLM_MEMORY_REQUEST", "12Gi")
VLLM_CPU_LIMIT = os.getenv("VLLM_CPU_LIMIT", VLLM_CPU_REQUEST)
VLLM_MEMORY_LIMIT = os.getenv("VLLM_MEMORY_LIMIT", VLLM_MEMORY_REQUEST)
TRAIN_GPU_COUNT = VLLM_GPU_COUNT - 1   # GPUs 1..N for DDP training
VLLM_GPU_MEM_UTIL = os.getenv("VLLM_GPU_MEM_UTIL", "0.4")
VLLM_LOAD_FORMAT = os.getenv("VLLM_LOAD_FORMAT", "auto")
RUN_TRAINING = os.getenv("RUN_GRPO_TRAINING", "true").lower() == "true"

who = subprocess.run(["oc", "whoami"], text=True, capture_output=True, check=False)
if who.returncode != 0:
    raise RuntimeError("`oc` is not authenticated. Run `oc login` and retry.")

# Keep kubernetes client objects for downstream cells.
# Use in-cluster config first to avoid anonymous auth paths.
try:
    k8s_config.load_incluster_config()
except Exception:
    k8s_config.load_kube_config()
api_client = k8s.ApiClient()
apps = k8s.AppsV1Api(api_client)
core = k8s.CoreV1Api(api_client)

service_host = f"{VLLM_NAME}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"
base_url = f"{service_root}/v1"
health_url = f"{service_root}/health"

print(f"User: {who.stdout.strip()}")
print(f"Namespace: {NAMESPACE}")
print(f"vLLM service: {service_host}")
print(f"Image: {IMAGE}")
print(f"Total GPUs: {VLLM_GPU_COUNT} (vLLM: GPU 0, training DDP: GPUs 1-{VLLM_GPU_COUNT-1})")
print(f"Resources: cpu={VLLM_CPU_REQUEST}, memory={VLLM_MEMORY_REQUEST}")
print(f"Load-format: {VLLM_LOAD_FORMAT}")
print(f"Run GRPO training: {RUN_TRAINING}")

Namespace: grpoxtrainer
vLLM service: grpo-vllm-rollout.grpoxtrainer.svc.cluster.local
Image: image-registry.openshift-image-registry.svc:5000/grpoxtrainer/vllm-async-init-poc:async-init-poc-20260519-150604
Total GPUs: 3 (vLLM: GPU 0, training DDP: GPUs 1-2)
Load-format: auto
Run GRPO training: True


## Deploy vLLM (3 GPUs, real weights, NCCL weight transfer)

In [4]:
cache_dir = "/tmp/hf-cache"
home_dir = "/tmp/vllm-home"
flashinfer_dir = "/tmp/flashinfer-cache"

deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": VLLM_NAME}},
        "template": {
            "metadata": {"labels": {"app": VLLM_NAME}},
            "spec": {
                "containers": [
                    {
                        "name": "vllm",
                        "image": IMAGE,
                        "args": [
                            MODEL_ID,
                            "--host", "0.0.0.0",
                            "--port", "8000",
                            "--gpu-memory-utilization", VLLM_GPU_MEM_UTIL,
                            "--max-model-len", "4096",
                            "--enforce-eager",
                            "--load-format", VLLM_LOAD_FORMAT,
                            "--weight-transfer-config",
                            json.dumps({"backend": WEIGHT_TRANSFER_BACKEND}),
                        ],
                        "ports": [{"containerPort": 8000, "name": "http"}],
                        "env": [
                            {"name": "HF_TOKEN", "value": os.getenv("HF_TOKEN", "")},
                            {"name": "HOME", "value": home_dir},
                            {"name": "HF_HOME", "value": cache_dir},
                            {"name": "HUGGINGFACE_HUB_CACHE", "value": cache_dir},
                            {"name": "TRANSFORMERS_CACHE", "value": cache_dir},
                            {"name": "XDG_CACHE_HOME", "value": "/tmp"},
                            {"name": "VLLM_CACHE_ROOT", "value": "/tmp/vllm-cache"},
                            {"name": "VLLM_CONFIG_ROOT", "value": "/tmp/vllm-config"},
                            {"name": "FLASHINFER_WORKSPACE_DIR", "value": flashinfer_dir},
                            {"name": "VLLM_SERVER_DEV_MODE", "value": "1"},
                            {"name": "VLLM_ALLOW_INSECURE_SERIALIZATION", "value": "1"},
                        ],
                        "resources": {
                            "requests": {
                                "cpu": VLLM_CPU_REQUEST,
                                "memory": VLLM_MEMORY_REQUEST,
                                "nvidia.com/gpu": str(VLLM_GPU_COUNT),
                            },
                            "limits": {
                                "cpu": VLLM_CPU_LIMIT,
                                "memory": VLLM_MEMORY_LIMIT,
                                "nvidia.com/gpu": str(VLLM_GPU_COUNT),
                            },
                        },
                        "volumeMounts": [
                            {"name": "hf-cache", "mountPath": cache_dir},
                            {"name": "vllm-home", "mountPath": home_dir},
                            {"name": "flashinfer-cache", "mountPath": flashinfer_dir},
                        ],
                    }
                ],
                "volumes": [
                    {"name": "hf-cache", "emptyDir": {}},
                    {"name": "vllm-home", "emptyDir": {}},
                    {"name": "flashinfer-cache", "emptyDir": {}},
                ],
            },
        },
    },
}
service = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "selector": {"app": VLLM_NAME},
        "ports": [{"port": 8000, "targetPort": 8000, "name": "http"}],
    },
}

try:
    apps.create_namespaced_deployment(namespace=NAMESPACE, body=deployment)
except Exception:
    apps.patch_namespaced_deployment(name=VLLM_NAME, namespace=NAMESPACE, body=deployment)

try:
    core.create_namespaced_service(namespace=NAMESPACE, body=service)
except Exception:
    core.patch_namespaced_service(name=VLLM_NAME, namespace=NAMESPACE, body=service)

print(f"Created or updated Deployment/Service for {VLLM_NAME}")

Created or updated Deployment/Service for grpo-vllm-rollout


## Wait for vLLM health

In [5]:
print("Waiting for deployment rollout to finish...")
for _ in range(60):
    dep = apps.read_namespaced_deployment(name=VLLM_NAME, namespace=NAMESPACE)
    ready = dep.status.ready_replicas or 0
    updated = dep.status.updated_replicas or 0
    desired = dep.spec.replicas or 1
    if ready >= desired and updated >= desired:
        print(f"  Rollout complete: {ready}/{desired} ready, {updated} updated")
        break
    print(f"  Rollout in progress: {ready}/{desired} ready, {updated} updated")
    time.sleep(10)
else:
    raise RuntimeError("Deployment rollout did not complete in time")

print("Waiting for vLLM /health endpoint...")
for _ in range(90):
    try:
        if requests.get(health_url, timeout=5).status_code == 200:
            print("vLLM health check passed")
            break
    except Exception:
        pass
    time.sleep(10)
else:
    raise RuntimeError("vLLM did not become healthy in time")

Waiting for deployment rollout to finish...
  Rollout complete: 1/1 ready, 1 updated
Waiting for vLLM /health endpoint...
vLLM health check passed


## GRPO multi-GPU training + weight sync (inside vLLM pod)

Copies `scripts/grpo_vllm_train_sync.py` into the pod, then runs two phases:

- **Phase A** — `torchrun --nproc_per_node=2` on GPUs 1 & 2 (`--train-only`): baseline eval, DDP GRPO training, save checkpoint
- **Phase B** — Plain python on GPUs 0 & 1 (`--sync-only`): load checkpoint, NCCL sync to vLLM, post-sync eval + Prometheus

In [6]:
if not RUN_TRAINING:
    print("Skipping GRPO training. Set RUN_GRPO_TRAINING=true to run.")
    result = None
else:
    vllm_pods = core.list_namespaced_pod(
        namespace=NAMESPACE, label_selector=f"app={VLLM_NAME}"
    )
    vllm_pod_name = vllm_pods.items[0].metadata.name
    print(f"Target vLLM pod: {vllm_pod_name}")

    script_src = os.path.join(
        os.path.dirname(os.path.abspath(".")),
        "scripts",
        "grpo_vllm_train_sync.py",
    )
    if not os.path.exists(script_src):
        script_src = "/opt/app-root/src/grpo_vllm_train_sync.py"
    if not os.path.exists(script_src):
        raise FileNotFoundError(
            "grpo_vllm_train_sync.py not found — copy scripts/ to workbench or run from repo root"
        )

    remote_script = "/tmp/grpo_vllm_train_sync.py"
    subprocess.run(
        ["oc", "cp", script_src, f"{NAMESPACE}/{vllm_pod_name}:{remote_script}"],
        check=True,
    )
    print(f"Copied {script_src} -> {vllm_pod_name}:{remote_script}")

    subprocess.run(
        ["oc", "exec", vllm_pod_name, "-n", NAMESPACE, "--",
         "pip", "install", "-q", "trl", "datasets", "accelerate"],
        check=True,
    )

    train_gpus = ",".join(str(i) for i in range(1, VLLM_GPU_COUNT))  # "1,2"
    sync_gpus = "0,1"

    def _stream_exec(cmd_desc, bash_cmd):
        """Run oc exec with streaming output, return combined stdout."""
        print(f"\n--- {cmd_desc} (streaming logs) ---\n")
        proc = subprocess.Popen(
            ["oc", "exec", vllm_pod_name, "-n", NAMESPACE, "--",
             "bash", "-c", bash_cmd],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines = []
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        proc.wait()
        return proc.returncode, "".join(lines)

    # Phase A: multi-GPU GRPO training via torchrun
    rc_a, out_a = _stream_exec(
        f"Phase A: torchrun DDP training on GPUs [{train_gpus}]",
        f"CUDA_VISIBLE_DEVICES={train_gpus} NCCL_DEBUG=WARN "
        f"torchrun --nproc_per_node={TRAIN_GPU_COUNT} "
        f"{remote_script} --train-only",
    )
    if rc_a != 0:
        raise RuntimeError(f"Phase A (training) failed with exit {rc_a}")
    print("\nPhase A complete — checkpoint saved.")

    # Phase B: weight sync from checkpoint to vLLM
    rc_b, out_b = _stream_exec(
        f"Phase B: weight sync (GPU 1 -> vLLM GPU 0)",
        f"CUDA_VISIBLE_DEVICES={sync_gpus} NCCL_DEBUG=WARN "
        f"python3 -u {remote_script} --sync-only",
    )
    if rc_b != 0:
        raise RuntimeError(f"Phase B (sync) failed with exit {rc_b}")

    class _Result:
        pass
    result = _Result()
    result.stdout = out_a + out_b
    result.returncode = 0

    print("\nMulti-GPU GRPO training + weight sync completed successfully.")

Target vLLM pod: grpo-vllm-rollout-76b4c5dd6f-lklzj


Copied /opt/app-root/src/grpo_vllm_train_sync.py -> grpo-vllm-rollout-76b4c5dd6f-lklzj:/tmp/grpo_vllm_train_sync.py



--- Phase A: torchrun DDP training on GPUs [1,2] (streaming logs) ---



W0520 08:01:17.177000 736 torch/distributed/run.py:851] 


W0520 08:01:17.177000 736 torch/distributed/run.py:851] *****************************************


...

  GRPO train time:      176.8s


  Weight sync time:     0.46s


__RESULTS_JSON__{"pre_eval": {"label": "BEFORE_TRAINING", "n": 10, "correct": 1, "accuracy": 0.1, "avg_latency_s": 0.1695, "results": [{"i": 0, "question": "Janet\u2019s ducks lay 16 eggs per day. She eats three for breakfast every morning an", "ref": "18", "pred": "320", "correct": false, "latency_s": 0.663, "snippet": "#### 320"}, {"i": 1, "question": "A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bol", "ref": "3", "pred": "4", "correct": false, "latency_s": 0.086, "snippet": "#### 4"}, {"i": 2, "question": "Josh decides to try flipping a house.  He buys a house for $80,000 and then puts", "ref": "70000", "pred": "25000", "correct": false, "latency_s": 0.147, "snippet": "#### 25000"}, {"i": 3, "question": "James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  ", "ref": "540", "pred": "180", "correct": false, "latency_s": 0.116, "snippet": "#### 180"}, {"i": 4, "question": "Every day, Wendi feeds each of her chickens three cups 

=== GRPO TRAINING + WEIGHT SYNC COMPLETE ===



Multi-GPU GRPO training + weight sync completed successfully.


## Results summary (accuracy + Prometheus)

In [7]:
if RUN_TRAINING and result is not None and result.stdout:
    marker = "__RESULTS_JSON__"
    results_data = None
    for line in result.stdout.splitlines():
        if marker in line:
            results_data = json.loads(line.split(marker, 1)[1])
            break

    if results_data:
        from IPython.display import HTML, display

        pre = results_data["pre_eval"]
        post = results_data["post_eval"]
        pre_m = results_data["pre_metrics"]
        post_m = results_data["post_metrics"]
        train_s = results_data["train_elapsed_s"]
        sync_s = results_data["sync_elapsed_s"]
        r0 = results_data["reward_mean_first"]
        r1 = results_data["reward_mean_last"]
        train_ws = results_data.get("train_world_size", 1)

        acc_delta = post["accuracy"] - pre["accuracy"]
        acc_color = "green" if acc_delta > 0 else ("red" if acc_delta < 0 else "black")

        eval_rows = ""
        for label, ev in [("Before training", pre), ("After training + sync", post)]:
            eval_rows += (
                f"<tr><td>{label}</td>"
                f"<td>{ev['correct']}/{ev['n']}</td>"
                f"<td>{ev['accuracy']:.1%}</td>"
                f"<td>{ev['avg_latency_s']:.3f}s</td></tr>\n"
            )

        prom_rows = ""
        for k in sorted(set(pre_m.keys()) | set(post_m.keys())):
            short = k.replace("vllm:", "")
            bv, av = pre_m.get(k, 0), post_m.get(k, 0)
            prom_rows += (
                f"<tr><td>{short}</td><td>{bv:.4f}</td>"
                f"<td>{av:.4f}</td><td>{av - bv:+.4f}</td></tr>\n"
            )

        detail_rows = ""
        for phase, ev in [("Before", pre), ("After", post)]:
            for row in ev.get("results", [])[:5]:
                mark = "OK" if row["correct"] else "MISS"
                detail_rows += (
                    f"<tr><td>{phase}</td><td>{mark}</td>"
                    f"<td>{row['question']}</td>"
                    f"<td>{row['ref']}</td><td>{row['pred']}</td></tr>\n"
                )

        html = f"""
        <h3>GRPO Multi-GPU Training + vLLM Weight Sync — Results</h3>
        <p><b>Training GPUs:</b> {train_ws} (DDP) &nbsp;|&nbsp;
        <b>GRPO reward/mean:</b> {r0:.4f} &rarr; {r1:.4f} &nbsp;|&nbsp;
        <b>Train:</b> {train_s}s &nbsp;|&nbsp; <b>Weight sync:</b> {sync_s}s</p>
        <h4>GSM8K accuracy (10 test questions via vLLM)</h4>
        <table border="1" cellpadding="6" style="border-collapse:collapse;font-family:monospace;">
        <tr style="background:#f0f0f0"><th>Phase</th><th>Correct</th><th>Accuracy</th><th>Avg latency</th></tr>
        {eval_rows}
        <tr><td colspan="4" style="color:{acc_color};font-weight:bold;">
        Accuracy delta: {acc_delta:+.1%}</td></tr>
        </table>
        <br/>
        <h4>Sample questions (first 5)</h4>
        <table border="1" cellpadding="6" style="border-collapse:collapse;font-family:monospace;font-size:12px;">
        <tr style="background:#f0f0f0"><th>Phase</th><th></th><th>Question</th><th>Ref</th><th>Pred</th></tr>
        {detail_rows}
        </table>
        <br/>
        <h4>Prometheus vLLM metrics</h4>
        <table border="1" cellpadding="6" style="border-collapse:collapse;font-family:monospace;">
        <tr style="background:#f0f0f0"><th>Metric</th><th>Before</th><th>After</th><th>Delta</th></tr>
        {prom_rows}
        </table>
        """
        display(HTML(html))
    else:
        print("Could not parse __RESULTS_JSON__ from script output.")
else:
    print("No training results to display.")

## Cross-node validation (KFT trainer pod + deployed vLLM)

This section runs the full cross-node cycle in one namespace (default `kapil-test`) using:

- one **vLLM serving Deployment** (default: `grpo-vllm-rollout`) pinned to `SERVING_NODE`
- one **KFT TrainJob** (default: `grpo-cross-node-validate`) pinned to `TRAINER_NODE`
- vLLM native NCCL weight-sync lifecycle (`init -> pause -> start -> update -> finish -> resume`)

By default, serving and trainer nodes are auto-discovered from available GPU workers.
You can still override with `PHASE2_SERVING_NODE` and `PHASE2_TRAINER_NODE` if needed.

It intentionally uses a temporary permissive ingress rule from the vLLM pod to the trainer pod to unblock NCCL data-channel ports for PoC speed.

## Quick start

Use this section as a manual cross-node validation runbook.

### Recommended run order

1. Set environment overrides in this section (namespace, optional node pins, image).
2. Run **Preflight checks** cell (auth, CRDs, runtime, RBAC verbs, image-pull viability).
3. Run **RBAC/bootstrap** cell only if preflight says permissions are missing.
4. Run **Cross-node config** cell (resolves serving/trainer nodes and enforces split placement).
5. Run **Cross-node execution** cell (deploy vLLM, apply all-ports policy, run TrainJob, validate markers).
6. Re-run **Cross-node execution** cell once to confirm repeatability.

### Optional overrides

- `PHASE2_NAMESPACE`: namespace for both vLLM and TrainJob
- `PHASE2_TEST_MODE`: `cross_node` (default) or `single_node`
- `PHASE2_SERVING_NODE`: manual serving-node pin
- `PHASE2_TRAINER_NODE`: manual trainer-node pin
- `PHASE2_VLLM_NAME`: deployment/service name (default `grpo-vllm-rollout`)
- `PHASE2_TRAINJOB_NAME`: TrainJob name (default `grpo-cross-node-validate`)
- `PHASE2_NETWORKPOLICY_NAME`: policy name (default `vllm-to-trainer-nccl-allports`)
- `PHASE2_VLLM_GPU_COUNT`: GPUs requested by vLLM deployment
- `PHASE2_VLLM_CPU_REQUEST` / `PHASE2_VLLM_MEMORY_REQUEST`: serving pod sizing
- `PHASE2_TRAINER_CPU_REQUEST` / `PHASE2_TRAINER_MEMORY_REQUEST`: trainer pod sizing
- `PHASE2_VLLM_IMAGE`: vLLM image for serving and trainer
- `PHASE2_BOOTSTRAP_RBAC`: set to `true` to apply notebook SA role bindings and image-puller grants

### Known failure recovery

- `oc` not authenticated: run `oc login`, then rerun preflight.
- Missing RBAC verbs in `kapil-test`: run bootstrap cell with `PHASE2_BOOTSTRAP_RBAC=true`.
- Init stuck at `initializing` or `409 already in progress`: rerun execution cell (it recreates vLLM/TrainJob/NetworkPolicy) and inspect printed status snapshots.
- Serving and trainer end up on same node in `cross_node` mode: set `PHASE2_SERVING_NODE` and `PHASE2_TRAINER_NODE` explicitly.

### Example

```python
import os
os.environ["PHASE2_NAMESPACE"] = "kapil-test"
os.environ["PHASE2_TEST_MODE"] = "cross_node"
os.environ["PHASE2_BOOTSTRAP_RBAC"] = "false"
# Optional pins:
# os.environ["PHASE2_SERVING_NODE"] = "ip-10-0-x-y..."
# os.environ["PHASE2_TRAINER_NODE"] = "ip-10-0-a-b..."
```

In [ ]:
import json
import os
import subprocess

NAMESPACE = os.getenv("PHASE2_NAMESPACE", os.getenv("NOTEBOOK_NAMESPACE", "kapil-test")).strip()
SOURCE_NS = os.getenv("PHASE2_SOURCE_IMAGE_NAMESPACE", "grpoxtrainer").strip()


def run_check(cmd):
    p = subprocess.run(cmd, text=True, capture_output=True, check=False)
    return {
        "cmd": " ".join(cmd),
        "ok": p.returncode == 0,
        "stdout": (p.stdout or "").strip(),
        "stderr": (p.stderr or "").strip(),
    }


who = run_check(["oc", "whoami"])
if not who["ok"]:
    raise RuntimeError("`oc` is not authenticated. Run `oc login` and retry.")

project = run_check(["oc", "project", "-q"])
current_project = project["stdout"] if project["ok"] else "<unknown>"

required_objects = {
    "trainjob_crd": run_check(["oc", "get", "crd", "trainjobs.trainer.kubeflow.org"]),
    "cluster_runtime": run_check(["oc", "get", "clustertrainingruntime", "torch-distributed"]),
}

missing_objects = [name for name, out in required_objects.items() if not out["ok"]]
if missing_objects:
    details = {name: required_objects[name]["stderr"] for name in missing_objects}
    raise RuntimeError(
        "Missing required cluster objects for cross-node run: "
        + json.dumps(details, indent=2)
    )

verbs_to_check = [
    ("create", "deployments.apps"),
    ("get", "deployments.apps"),
    ("delete", "deployments.apps"),
    ("create", "services"),
    ("get", "services"),
    ("delete", "services"),
    ("create", "networkpolicies.networking.k8s.io"),
    ("get", "networkpolicies.networking.k8s.io"),
    ("delete", "networkpolicies.networking.k8s.io"),
    ("create", "trainjobs.trainer.kubeflow.org"),
    ("get", "trainjobs.trainer.kubeflow.org"),
    ("delete", "trainjobs.trainer.kubeflow.org"),
    ("get", "pods"),
    ("list", "pods"),
    ("get", "pods/log"),
]

permission_results = {}
missing_permissions = []
for verb, resource in verbs_to_check:
    check = run_check(["oc", "auth", "can-i", verb, resource, "-n", NAMESPACE])
    allowed = check["ok"] and check["stdout"].strip().lower() == "yes"
    key = f"{verb} {resource}"
    permission_results[key] = allowed
    if not allowed:
        missing_permissions.append(key)

hostname = os.getenv("HOSTNAME", "")
sa_name = os.getenv("NOTEBOOK_SA", "").strip()
if not sa_name and hostname:
    sa_lookup = run_check(["oc", "get", "pod", hostname, "-n", NAMESPACE, "-o", "jsonpath={.spec.serviceAccountName}"])
    if sa_lookup["ok"] and sa_lookup["stdout"]:
        sa_name = sa_lookup["stdout"]

image_pull_checks = {}
for subject_name in ["default", sa_name]:
    if not subject_name:
        continue
    check = run_check([
        "oc",
        "auth",
        "can-i",
        "get",
        "imagestreams/layers",
        "-n",
        SOURCE_NS,
        f"--as=system:serviceaccount:{NAMESPACE}:{subject_name}",
    ])
    image_pull_checks[subject_name] = check["ok"] and check["stdout"].strip().lower() == "yes"

summary = {
    "user": who["stdout"],
    "current_project": current_project,
    "target_namespace": NAMESPACE,
    "source_image_namespace": SOURCE_NS,
    "required_objects_ok": not missing_objects,
    "missing_permissions": missing_permissions,
    "permissions": permission_results,
    "notebook_service_account": sa_name or "<unknown>",
    "image_pull_checks": image_pull_checks,
}

os.environ["PHASE2_NAMESPACE"] = NAMESPACE
if sa_name:
    os.environ["NOTEBOOK_SA"] = sa_name
os.environ["PHASE2_PRECHECK_MISSING_PERMISSIONS"] = json.dumps(missing_permissions)
os.environ["PHASE2_PRECHECK_IMAGE_PULL"] = json.dumps(image_pull_checks)

print("__PHASE2_PREFLIGHT_SUMMARY__")
print(json.dumps(summary, indent=2))

if missing_permissions:
    print("\nMissing namespace permissions detected. Run bootstrap cell with PHASE2_BOOTSTRAP_RBAC=true, then rerun preflight.")
else:
    print("\nNamespace RBAC checks passed for cross-node execution.")

if image_pull_checks and not any(image_pull_checks.values()):
    print("Image pull access from source namespace appears missing. Bootstrap cell can add image-puller grants.")

In [ ]:
import os
import subprocess
import textwrap

BOOTSTRAP = os.getenv("PHASE2_BOOTSTRAP_RBAC", "false").strip().lower() == "true"
NAMESPACE = os.getenv("PHASE2_NAMESPACE", os.getenv("NOTEBOOK_NAMESPACE", "kapil-test")).strip()
SOURCE_NS = os.getenv("PHASE2_SOURCE_IMAGE_NAMESPACE", "grpoxtrainer").strip()
SA_NAME = os.getenv("NOTEBOOK_SA", "").strip()

if not SA_NAME:
    host = os.getenv("HOSTNAME", "")
    if host:
        lookup = subprocess.run(
            ["oc", "get", "pod", host, "-n", NAMESPACE, "-o", "jsonpath={.spec.serviceAccountName}"],
            text=True,
            capture_output=True,
            check=False,
        )
        if lookup.returncode == 0:
            SA_NAME = (lookup.stdout or "").strip()

if not BOOTSTRAP:
    print("PHASE2_BOOTSTRAP_RBAC is false; skipping RBAC/image-puller bootstrap.")
    print("Set os.environ['PHASE2_BOOTSTRAP_RBAC'] = 'true' and rerun this cell when preflight reports missing permissions.")
else:
    if not SA_NAME:
        raise RuntimeError("Could not resolve notebook service account name for RoleBinding.")

    print(f"Bootstrapping RBAC in namespace {NAMESPACE} for service account {SA_NAME}")

    manifest = textwrap.dedent(
        f"""
        apiVersion: rbac.authorization.k8s.io/v1
        kind: Role
        metadata:
          name: phase2-cross-node-notebook-runner
          namespace: {NAMESPACE}
        rules:
          - apiGroups: [""]
            resources: ["pods", "pods/log"]
            verbs: ["get", "list", "watch", "delete"]
          - apiGroups: ["apps"]
            resources: ["deployments"]
            verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
          - apiGroups: [""]
            resources: ["services"]
            verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
          - apiGroups: ["networking.k8s.io"]
            resources: ["networkpolicies"]
            verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
          - apiGroups: ["trainer.kubeflow.org"]
            resources: ["trainjobs"]
            verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
        ---
        apiVersion: rbac.authorization.k8s.io/v1
        kind: RoleBinding
        metadata:
          name: phase2-cross-node-notebook-runner
          namespace: {NAMESPACE}
        roleRef:
          apiGroup: rbac.authorization.k8s.io
          kind: Role
          name: phase2-cross-node-notebook-runner
        subjects:
          - kind: ServiceAccount
            name: {SA_NAME}
            namespace: {NAMESPACE}
        """
    )

    apply = subprocess.run(
        ["oc", "apply", "-f", "-"],
        input=manifest,
        text=True,
        capture_output=True,
        check=False,
    )
    print((apply.stdout or "").strip())
    if apply.returncode != 0:
        raise RuntimeError((apply.stderr or "").strip() or "Failed to apply RBAC manifest")

    for cmd in [
        ["oc", "policy", "add-role-to-group", "system:image-puller", f"system:serviceaccounts:{NAMESPACE}", "-n", SOURCE_NS],
        ["oc", "policy", "add-role-to-user", "system:image-puller", f"system:serviceaccount:{NAMESPACE}:{SA_NAME}", "-n", SOURCE_NS],
    ]:
        out = subprocess.run(cmd, text=True, capture_output=True, check=False)
        if out.stdout:
            print(out.stdout.strip())
        if out.returncode != 0:
            print((out.stderr or "").strip())
            print("Warning: image-puller grant command failed; verify cluster permissions and source namespace.")

    print("Bootstrap completed. Rerun preflight cell to confirm permissions are now satisfied.")

In [ ]:
import json
import os
import subprocess
import textwrap
import time

NAMESPACE = os.getenv("PHASE2_NAMESPACE", os.getenv("NOTEBOOK_NAMESPACE", "kapil-test")).strip()
TEST_MODE = os.getenv("PHASE2_TEST_MODE", "cross_node").strip().lower()
if TEST_MODE not in {"cross_node", "single_node"}:
    TEST_MODE = "cross_node"

SERVING_NODE = os.getenv("PHASE2_SERVING_NODE", "").strip()
TRAINER_NODE = os.getenv("PHASE2_TRAINER_NODE", "").strip()

VLLM_NAME = os.getenv("PHASE2_VLLM_NAME", "grpo-vllm-rollout").strip()
TRAINJOB_NAME = os.getenv("PHASE2_TRAINJOB_NAME", "grpo-cross-node-validate").strip()
NP_NAME = os.getenv("PHASE2_NETWORKPOLICY_NAME", "vllm-to-trainer-nccl-allports").strip()

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
VLLM_IMAGE = os.getenv(
    "PHASE2_VLLM_IMAGE",
    "image-registry.openshift-image-registry.svc:5000/grpoxtrainer/vllm-async-init-poc@sha256:996350ac0f2320755df840a456552ec8ad7cc22ec3178034ad5dba75711daf1d",
)
VLLM_GPU_COUNT = int(os.getenv("PHASE2_VLLM_GPU_COUNT", "1" if TEST_MODE == "cross_node" else "3"))
VLLM_CPU_REQUEST = os.getenv("PHASE2_VLLM_CPU_REQUEST", "2")
VLLM_MEMORY_REQUEST = os.getenv("PHASE2_VLLM_MEMORY_REQUEST", "12Gi")
VLLM_CPU_LIMIT = os.getenv("PHASE2_VLLM_CPU_LIMIT", VLLM_CPU_REQUEST)
VLLM_MEMORY_LIMIT = os.getenv("PHASE2_VLLM_MEMORY_LIMIT", VLLM_MEMORY_REQUEST)
TRAINER_CPU_REQUEST = os.getenv("PHASE2_TRAINER_CPU_REQUEST", "4")
TRAINER_MEMORY_REQUEST = os.getenv("PHASE2_TRAINER_MEMORY_REQUEST", "24Gi")
TRAINER_CPU_LIMIT = os.getenv("PHASE2_TRAINER_CPU_LIMIT", TRAINER_CPU_REQUEST)
TRAINER_MEMORY_LIMIT = os.getenv("PHASE2_TRAINER_MEMORY_LIMIT", TRAINER_MEMORY_REQUEST)


def _parse_int(value, default=0):
    try:
        return int(str(value))
    except (TypeError, ValueError):
        return default


def _parse_cpu_m(value):
    s = str(value or "0").strip()
    if s.endswith("m"):
        return int(float(s[:-1] or 0))
    return int(float(s) * 1000)


def _parse_mem_mib(value):
    s = str(value or "0").strip()
    if not s:
        return 0
    units = {
        "Ki": 1 / 1024,
        "Mi": 1,
        "Gi": 1024,
        "Ti": 1024 * 1024,
        "K": 1000 / (1024 * 1024),
        "M": 1000 * 1000 / (1024 * 1024),
        "G": 1000 * 1000 * 1000 / (1024 * 1024),
        "T": 1000 * 1000 * 1000 * 1000 / (1024 * 1024),
    }
    for suffix, factor in units.items():
        if s.endswith(suffix):
            return int(float(s[: -len(suffix)] or 0) * factor)
    return int(float(s) / (1024 * 1024))


def _node_is_schedulable(node):
    if bool(node.get("spec", {}).get("unschedulable", False)):
        return False
    taints = node.get("spec", {}).get("taints", [])
    for taint in taints:
        if taint.get("effect") in {"NoSchedule", "NoExecute"}:
            return False
    return True

if not SERVING_NODE or not TRAINER_NODE:
    nodes = json.loads(
        subprocess.run(
            ["oc", "get", "nodes", "-o", "json"],
            text=True,
            capture_output=True,
            check=True,
        ).stdout
    ).get("items", [])
    pods = json.loads(
        subprocess.run(
            ["oc", "get", "pods", "-A", "-o", "json"],
            text=True,
            capture_output=True,
            check=True,
        ).stdout
    ).get("items", [])

    usage = {}
    for node in nodes:
        name = node["metadata"]["name"]
        alloc = node.get("status", {}).get("allocatable", {})
        ready = any(
            c.get("type") == "Ready" and c.get("status") == "True"
            for c in node.get("status", {}).get("conditions", [])
        )
        if not ready or not _node_is_schedulable(node):
            continue
        gpu_alloc = _parse_int(alloc.get("nvidia.com/gpu"), 0)
        cpu_alloc_m = _parse_cpu_m(alloc.get("cpu", "0"))
        mem_alloc_mib = _parse_mem_mib(alloc.get("memory", "0"))
        if gpu_alloc > 0 and cpu_alloc_m > 0 and mem_alloc_mib > 0:
            usage[name] = {
                "gpu_alloc": gpu_alloc,
                "gpu_used": 0,
                "cpu_alloc_m": cpu_alloc_m,
                "cpu_used_m": 0,
                "mem_alloc_mib": mem_alloc_mib,
                "mem_used_mib": 0,
            }

    for pod in pods:
        if pod.get("status", {}).get("phase") not in {"Running", "Pending"}:
            continue
        node_name = pod.get("spec", {}).get("nodeName")
        if node_name not in usage:
            continue

        pod_gpu = 0
        pod_cpu_m = 0
        pod_mem_mib = 0
        for c in pod.get("spec", {}).get("containers", []):
            reqs = c.get("resources", {}).get("requests", {})
            lims = c.get("resources", {}).get("limits", {})

            g = reqs.get("nvidia.com/gpu")
            if g is None:
                g = lims.get("nvidia.com/gpu")
            pod_gpu += _parse_int(g, 0)

            cpu = reqs.get("cpu")
            if cpu is None:
                cpu = lims.get("cpu")
            pod_cpu_m += _parse_cpu_m(cpu or "0")

            mem = reqs.get("memory")
            if mem is None:
                mem = lims.get("memory")
            pod_mem_mib += _parse_mem_mib(mem or "0")

        usage[node_name]["gpu_used"] += pod_gpu
        usage[node_name]["cpu_used_m"] += pod_cpu_m
        usage[node_name]["mem_used_mib"] += pod_mem_mib

    vllm_need = {
        "gpu": max(VLLM_GPU_COUNT, 1),
        "cpu_m": _parse_cpu_m(VLLM_CPU_REQUEST),
        "mem_mib": _parse_mem_mib(VLLM_MEMORY_REQUEST),
    }
    trainer_need = {
        "gpu": 1,
        "cpu_m": _parse_cpu_m(TRAINER_CPU_REQUEST),
        "mem_mib": _parse_mem_mib(TRAINER_MEMORY_REQUEST),
    }

    def _fits(name, need):
        node_use = usage[name]
        return (
            node_use["gpu_alloc"] - node_use["gpu_used"] >= need["gpu"]
            and node_use["cpu_alloc_m"] - node_use["cpu_used_m"] >= need["cpu_m"]
            and node_use["mem_alloc_mib"] - node_use["mem_used_mib"] >= need["mem_mib"]
        )

    serving_candidates = [name for name in usage if _fits(name, vllm_need)]
    trainer_candidates = [name for name in usage if _fits(name, trainer_need)]

    if not SERVING_NODE and serving_candidates:
        SERVING_NODE = serving_candidates[0]

    if not TRAINER_NODE:
        if TEST_MODE == "single_node":
            TRAINER_NODE = SERVING_NODE
        else:
            TRAINER_NODE = next((n for n in trainer_candidates if n != SERVING_NODE), "")
            if not TRAINER_NODE and trainer_candidates:
                TRAINER_NODE = trainer_candidates[0]

    NODE_CAPACITY_SUMMARY = usage
    SERVING_NODE_CANDIDATES = serving_candidates
    TRAINER_NODE_CANDIDATES = trainer_candidates
if not SERVING_NODE:
    raise RuntimeError(
        "Could not resolve serving node with enough free GPU/CPU/memory. "
        "Set PHASE2_SERVING_NODE manually or lower PHASE2_VLLM_* resource requests."
    )
if not TRAINER_NODE:
    raise RuntimeError(
        "Could not resolve trainer node with enough free GPU/CPU/memory. "
        "Set PHASE2_TRAINER_NODE manually or lower PHASE2_TRAINER_* resource requests."
    )

if SERVING_NODE in NODE_CAPACITY_SUMMARY and not (
    NODE_CAPACITY_SUMMARY[SERVING_NODE]["gpu_alloc"] - NODE_CAPACITY_SUMMARY[SERVING_NODE]["gpu_used"] >= max(VLLM_GPU_COUNT, 1)
):
    print("WARNING: selected serving node appears GPU-constrained:", SERVING_NODE)

if TRAINER_NODE in NODE_CAPACITY_SUMMARY and not (
    NODE_CAPACITY_SUMMARY[TRAINER_NODE]["gpu_alloc"] - NODE_CAPACITY_SUMMARY[TRAINER_NODE]["gpu_used"] >= 1
):
    print("WARNING: selected trainer node appears GPU-constrained:", TRAINER_NODE)

if TEST_MODE == "cross_node" and SERVING_NODE == TRAINER_NODE:
    raise RuntimeError(
        "cross_node mode requires different serving and trainer nodes. "
        "Set PHASE2_SERVING_NODE and PHASE2_TRAINER_NODE explicitly."
    )
if TEST_MODE == "single_node":
    TRAINER_NODE = SERVING_NODE

missing_permissions = json.loads(os.getenv("PHASE2_PRECHECK_MISSING_PERMISSIONS", "[]"))
if missing_permissions:
    print("WARNING: preflight reported missing permissions:")
    for perm in missing_permissions:
        print(" -", perm)
    print("Run bootstrap cell with PHASE2_BOOTSTRAP_RBAC=true, then rerun preflight.")

print(f"Namespace    : {NAMESPACE}")
print(f"Mode         : {TEST_MODE}")
print(f"Serving node : {SERVING_NODE}")
print(f"Trainer node : {TRAINER_NODE}")
print(f"vLLM name    : {VLLM_NAME}")
print(f"vLLM GPUs    : {VLLM_GPU_COUNT}")
print(f"vLLM image   : {VLLM_IMAGE}")
print(f"vLLM request : cpu={VLLM_CPU_REQUEST}, memory={VLLM_MEMORY_REQUEST}, gpu={VLLM_GPU_COUNT}")
print(f"Trainer req  : cpu={TRAINER_CPU_REQUEST}, memory={TRAINER_MEMORY_REQUEST}, gpu=1")
print("Serving candidates:", SERVING_NODE_CANDIDATES)
print("Trainer candidates:", TRAINER_NODE_CANDIDATES)

In [ ]:
def run_cmd(cmd, input_text=None, check=True, capture=True):
    print("$", " ".join(cmd))
    p = subprocess.run(
        cmd,
        input=input_text,
        text=True,
        capture_output=capture,
        check=False,
    )
    if capture and p.stdout:
        print(p.stdout.strip())
    if p.returncode != 0 and p.stderr:
        print(p.stderr.strip())
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p


def wait_for_vllm_http_ready(namespace, pod_name, timeout_s=900):
    probe = textwrap.dedent(
        """
        import json
        import requests
        import time

        base = "http://localhost:8000"
        deadline = time.time() + __TIMEOUT__
        last_error = ""

        while time.time() < deadline:
            try:
                health = requests.get(f"{base}/health", timeout=20)
                world = requests.get(f"{base}/get_world_size", timeout=20)
                if health.status_code == 200 and world.status_code == 200:
                    print("__VLLM_READY__" + json.dumps({
                        "health_status": health.status_code,
                        "world_size": world.json(),
                    }))
                    raise SystemExit(0)
                last_error = f"health={health.status_code} world={world.status_code}"
            except Exception as exc:  # noqa: BLE001
                last_error = str(exc)
            time.sleep(5)

        raise RuntimeError(f"vLLM API did not become ready in time: {last_error}")
        """
    ).replace("__TIMEOUT__", str(int(timeout_s)))

    run_cmd([
        "oc", "exec", "-n", namespace, pod_name, "--", "python3", "-c", probe
    ])

vllm_manifest = textwrap.dedent(
    """
    apiVersion: apps/v1
    kind: Deployment
    metadata:
      name: __VLLM_NAME__
      namespace: __NS__
      labels:
        app: __VLLM_NAME__
    spec:
      replicas: 1
      selector:
        matchLabels:
          app: __VLLM_NAME__
      template:
        metadata:
          labels:
            app: __VLLM_NAME__
        spec:
          nodeSelector:
            kubernetes.io/hostname: __SERVING_NODE__
          containers:
            - name: vllm
              image: __VLLM_IMAGE__
              args:
                - __MODEL_NAME__
                - --host
                - 0.0.0.0
                - --port
                - "8000"
                - --max-model-len
                - "4096"
                - --gpu-memory-utilization
                - "0.45"
                - --enforce-eager
                - --load-format
                - dummy
                - --weight-transfer-config
                - '{"backend":"nccl"}'
              env:
                - name: HF_TOKEN
                  value: ""
                - name: VLLM_SERVER_DEV_MODE
                  value: "1"
                - name: VLLM_ALLOW_INSECURE_SERIALIZATION
                  value: "1"
                - name: HOME
                  value: /tmp
                - name: HF_HOME
                  value: /tmp/hf-cache
                - name: HUGGINGFACE_HUB_CACHE
                  value: /tmp/hf-cache
                - name: TRANSFORMERS_CACHE
                  value: /tmp/hf-cache
                - name: XDG_CACHE_HOME
                  value: /tmp
                - name: USER
                  value: vllm
                - name: TORCHINDUCTOR_CACHE_DIR
                  value: /tmp/torchinductor
              ports:
                - name: http
                  containerPort: 8000
              resources:
                requests:
                  cpu: "__VLLM_CPU_REQUEST__"
                  memory: "__VLLM_MEMORY_REQUEST__"
                  nvidia.com/gpu: "__VLLM_GPU_COUNT__"
                limits:
                  cpu: "__VLLM_CPU_LIMIT__"
                  memory: "__VLLM_MEMORY_LIMIT__"
                  nvidia.com/gpu: "__VLLM_GPU_COUNT__"
              volumeMounts:
                - name: hf-cache
                  mountPath: /tmp/hf-cache
          volumes:
            - name: hf-cache
              emptyDir: {}
    ---
    apiVersion: v1
    kind: Service
    metadata:
      name: __VLLM_NAME__
      namespace: __NS__
      labels:
        app: __VLLM_NAME__
    spec:
      selector:
        app: __VLLM_NAME__
      ports:
        - name: http
          port: 8000
          targetPort: 8000
    ---
    apiVersion: networking.k8s.io/v1
    kind: NetworkPolicy
    metadata:
      name: __NP_NAME__
      namespace: __NS__
    spec:
      podSelector:
        matchLabels:
          phase2.grpo/sync-role: trainer-master
      policyTypes:
        - Ingress
      ingress:
        - from:
            - podSelector:
                matchLabels:
                  app: __VLLM_NAME__
    """
).replace("__VLLM_NAME__", VLLM_NAME).replace("__NS__", NAMESPACE).replace("__SERVING_NODE__", SERVING_NODE).replace("__VLLM_IMAGE__", VLLM_IMAGE).replace("__MODEL_NAME__", MODEL_NAME).replace("__NP_NAME__", NP_NAME).replace("__VLLM_GPU_COUNT__", str(VLLM_GPU_COUNT)).replace("__VLLM_CPU_REQUEST__", VLLM_CPU_REQUEST).replace("__VLLM_MEMORY_REQUEST__", VLLM_MEMORY_REQUEST).replace("__VLLM_CPU_LIMIT__", VLLM_CPU_LIMIT).replace("__VLLM_MEMORY_LIMIT__", VLLM_MEMORY_LIMIT)

trainjob_manifest = textwrap.dedent(
    """
    apiVersion: trainer.kubeflow.org/v1alpha1
    kind: TrainJob
    metadata:
      name: __TRAINJOB_NAME__
      namespace: __NS__
      labels:
        app.kubernetes.io/name: __TRAINJOB_NAME__
    spec:
      runtimeRef:
        apiGroup: trainer.kubeflow.org
        kind: ClusterTrainingRuntime
        name: torch-distributed
      trainer:
        image: __VLLM_IMAGE__
        numNodes: 1
        numProcPerNode: 1
        command:
          - bash
          - -lc
          - |
            set -euo pipefail
            cat > /tmp/cross_node_validate.py <<'PY'
            import json
            import os
            import threading
            import time

            import requests
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            from vllm.distributed.weight_transfer.nccl_engine import (
                NCCLTrainerSendWeightsArgs,
                NCCLWeightTransferEngine,
            )

            BASE_URL = os.environ["VLLM_BASE_URL"].rstrip("/")
            MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
            MASTER_PORT = int(os.environ.get("NCCL_MASTER_PORT", "29501"))
            POD_IP = os.environ["TRAINER_POD_IP"]

            def call_chat(prompt: str) -> str:
                payload = {
                    "model": MODEL_NAME,
                    "messages": [{"role": "user", "content": prompt}],
                    "max_tokens": 24,
                    "temperature": 0.0,
                }
                r = requests.post(f"{BASE_URL}/v1/chat/completions", json=payload, timeout=90)
                r.raise_for_status()
                body = r.json()
                return (body["choices"][0]["message"]["content"] or "").strip()

            def run_optimizer_steps(model, tokenizer):
                model.train()
                optim = torch.optim.AdamW(model.parameters(), lr=5e-6)
                prompts = [
                    "Solve: 9 + 11 = ?",
                    "Solve: 13 + 29 = ?",
                    "What is the capital of France?",
                    "State Newton's second law briefly.",
                ]
                losses = []
                for _ in range(2):
                    batch = tokenizer(
                        prompts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=64,
                    ).to("cuda:0")
                    outputs = model(**batch, labels=batch["input_ids"])
                    loss = outputs.loss
                    loss.backward()
                    optim.step()
                    optim.zero_grad(set_to_none=True)
                    losses.append(float(loss.item()))
                return losses

            def get_status_snapshot(timeout=20):
                resp = requests.get(f"{BASE_URL}/weight_transfer_status", timeout=timeout)
                resp.raise_for_status()
                return resp.json()

            def wait_for_init_ready(operation_id, timeout_s=300):
                snapshots = []
                deadline = time.time() + timeout_s
                while time.time() < deadline:
                    status = get_status_snapshot(timeout=30)
                    snapshots.append(status)
                    ops = status.get("operations", [])
                    op = next((o for o in ops if o.get("operation_id") == operation_id), None)
                    op_state = (op or {}).get("status", "")
                    if op_state in {"ready", "completed", "success"}:
                        return snapshots
                    if op_state in {"failed", "error"}:
                        raise RuntimeError(f"init operation entered terminal failure state: {op}")
                    time.sleep(3)
                raise RuntimeError(
                    "init operation stayed non-ready until timeout; "
                    f"operation_id={operation_id}, last_snapshot={snapshots[-1] if snapshots else {}}"
                )

            def sync_weights(train_model):
                ws = requests.get(f"{BASE_URL}/get_world_size", timeout=20).json()["world_size"]
                world_size = int(ws) + 1

                status_before = get_status_snapshot(timeout=20)
                running_before = status_before.get("running_operation_id")
                if running_before:
                    raise RuntimeError(
                        "stale running operation exists before init: "
                        f"running_operation_id={running_before}, status={status_before}"
                    )

                init_payload = {
                    "init_info": {
                        "master_address": POD_IP,
                        "master_port": MASTER_PORT,
                        "rank_offset": 1,
                        "world_size": world_size,
                    }
                }
                init_error = {"error": None}
                init_resp = {"status_code": None, "json": {}, "text": ""}

                def _do_init():
                    try:
                        resp = requests.post(
                            f"{BASE_URL}/init_weight_transfer_engine",
                            json=init_payload,
                            timeout=300,
                        )
                        init_resp["status_code"] = resp.status_code
                        init_resp["text"] = resp.text[:400]
                        try:
                            init_resp["json"] = resp.json()
                        except Exception:
                            init_resp["json"] = {}
                    except Exception as exc:
                        init_error["error"] = str(exc)

                t0 = time.perf_counter()
                init_thread = threading.Thread(target=_do_init, daemon=True)
                init_thread.start()
                group = NCCLWeightTransferEngine.trainer_init(
                    {
                        "master_address": POD_IP,
                        "master_port": MASTER_PORT,
                        "world_size": world_size,
                    }
                )
                init_thread.join()

                if init_error["error"]:
                    raise RuntimeError(f"init_weight_transfer_engine failed: {init_error['error']}")
                if init_resp["status_code"] == 409:
                    raise RuntimeError(
                        "init_weight_transfer_engine returned 409 already in progress: "
                        f"{init_resp['json'] or init_resp['text']}"
                    )
                if init_resp["status_code"] not in {200, 202}:
                    raise RuntimeError(
                        "init_weight_transfer_engine unexpected status: "
                        f"status={init_resp['status_code']} body={init_resp['json'] or init_resp['text']}"
                    )

                operation_id = init_resp["json"].get("operation_id") or get_status_snapshot().get("running_operation_id")
                snapshots = wait_for_init_ready(operation_id, timeout_s=300)

                requests.post(f"{BASE_URL}/pause", timeout=60).raise_for_status()
                requests.post(
                    f"{BASE_URL}/start_weight_update",
                    json={"is_checkpoint_format": True},
                    timeout=120,
                ).raise_for_status()

                names, dtype_names, shapes = [], [], []
                for name, p in train_model.named_parameters():
                    names.append(name)
                    dtype_names.append(str(p.dtype).split(".")[-1])
                    shapes.append(list(p.shape))

                update_payload = {
                    "update_info": {
                        "names": names,
                        "dtype_names": dtype_names,
                        "shapes": shapes,
                        "packed": True,
                    }
                }
                update_error = {"error": None}

                def _do_update():
                    try:
                        resp = requests.post(
                            f"{BASE_URL}/update_weights",
                            json=update_payload,
                            timeout=600,
                        )
                        resp.raise_for_status()
                    except Exception as exc:
                        update_error["error"] = str(exc)

                update_thread = threading.Thread(target=_do_update, daemon=True)
                update_thread.start()
                NCCLWeightTransferEngine.trainer_send_weights(
                    iterator=train_model.named_parameters(),
                    trainer_args=NCCLTrainerSendWeightsArgs(group=group, packed=True),
                )
                update_thread.join()
                if update_error["error"]:
                    raise RuntimeError(f"update_weights failed: {update_error['error']}")

                requests.post(f"{BASE_URL}/finish_weight_update", json={}, timeout=120).raise_for_status()
                requests.post(f"{BASE_URL}/resume", timeout=60).raise_for_status()
                return time.perf_counter() - t0, operation_id, snapshots

            def main():
                torch.cuda.set_device(0)
                dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
                result = {
                    "status": "unknown",
                    "model": MODEL_NAME,
                    "base_url": BASE_URL,
                    "trainer_pod_ip": POD_IP,
                    "trainer_node_name": os.environ.get("TRAINER_NODE_NAME", ""),
                    "expected_serving_node": os.environ.get("EXPECTED_SERVING_NODE", ""),
                    "expected_trainer_node": os.environ.get("EXPECTED_TRAINER_NODE", ""),
                    "master_port": MASTER_PORT,
                }

                result["health_status"] = requests.get(f"{BASE_URL}/health", timeout=30).status_code
                result["world_size_response"] = requests.get(f"{BASE_URL}/get_world_size", timeout=30).json()
                result["pre_sync_answer"] = call_chat("What is 2+2? Answer with just the number.")

                model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to("cuda:0")
                tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.padding_side = "left"

                t0 = time.perf_counter()
                result["optimizer_losses"] = run_optimizer_steps(model, tokenizer)
                result["optimizer_elapsed_s"] = round(time.perf_counter() - t0, 2)

                try:
                    sync_elapsed, operation_id, snapshots = sync_weights(model)
                    result["sync_status"] = "success"
                    result["sync_elapsed_s"] = round(sync_elapsed, 2)
                    result["sync_operation_id"] = operation_id
                    result["sync_status_snapshots"] = snapshots[-10:]
                except Exception as exc:  # noqa: BLE001
                    result["sync_status"] = "failed"
                    result["sync_error"] = str(exc)

                result["post_sync_answer"] = call_chat("What is 2+2? Answer with just the number.")
                result["status"] = "ok" if result.get("sync_status") == "success" else "failed"
                print("__CROSS_NODE_RESULT_JSON__" + json.dumps(result, default=str))

                if result["status"] != "ok":
                    raise SystemExit(2)

            if __name__ == "__main__":
                main()
            PY
            python3 /tmp/cross_node_validate.py
        env:
          - name: VLLM_BASE_URL
            value: http://__VLLM_NAME__.__NS__.svc.cluster.local:8000
          - name: MODEL_NAME
            value: __MODEL_NAME__
          - name: NCCL_MASTER_PORT
            value: "29501"
          - name: HOME
            value: /tmp
          - name: HF_HOME
            value: /tmp/hf-cache
          - name: HUGGINGFACE_HUB_CACHE
            value: /tmp/hf-cache
          - name: TRANSFORMERS_CACHE
            value: /tmp/hf-cache
          - name: XDG_CACHE_HOME
            value: /tmp
          - name: EXPECTED_SERVING_NODE
            value: __SERVING_NODE__
          - name: EXPECTED_TRAINER_NODE
            value: __TRAINER_NODE__
          - name: TRAINER_POD_IP
            valueFrom:
              fieldRef:
                fieldPath: status.podIP
          - name: TRAINER_NODE_NAME
            valueFrom:
              fieldRef:
                fieldPath: spec.nodeName
        resourcesPerNode:
          requests:
            cpu: "__TRAINER_CPU_REQUEST__"
            memory: "__TRAINER_MEMORY_REQUEST__"
            nvidia.com/gpu: "1"
          limits:
            cpu: "__TRAINER_CPU_LIMIT__"
            memory: "__TRAINER_MEMORY_LIMIT__"
            nvidia.com/gpu: "1"
      podTemplateOverrides:
        - targetJobs:
            - name: node
          metadata:
            labels:
              phase2.grpo/sync-role: trainer-master
          spec:
            nodeSelector:
              kubernetes.io/hostname: __TRAINER_NODE__
    """
).replace("__TRAINJOB_NAME__", TRAINJOB_NAME).replace("__NS__", NAMESPACE).replace("__VLLM_NAME__", VLLM_NAME).replace("__MODEL_NAME__", MODEL_NAME).replace("__VLLM_IMAGE__", VLLM_IMAGE).replace("__TRAINER_NODE__", TRAINER_NODE).replace("__SERVING_NODE__", SERVING_NODE).replace("__TRAINER_CPU_REQUEST__", TRAINER_CPU_REQUEST).replace("__TRAINER_MEMORY_REQUEST__", TRAINER_MEMORY_REQUEST).replace("__TRAINER_CPU_LIMIT__", TRAINER_CPU_LIMIT).replace("__TRAINER_MEMORY_LIMIT__", TRAINER_MEMORY_LIMIT)

run_cmd(["oc", "delete", "trainjob", TRAINJOB_NAME, "-n", NAMESPACE, "--ignore-not-found"]) 
run_cmd(["oc", "delete", "deployment", VLLM_NAME, "-n", NAMESPACE, "--ignore-not-found"]) 
run_cmd(["oc", "delete", "service", VLLM_NAME, "-n", NAMESPACE, "--ignore-not-found"]) 
run_cmd(["oc", "delete", "networkpolicy", NP_NAME, "-n", NAMESPACE, "--ignore-not-found"]) 

run_cmd(["oc", "apply", "-f", "-"], input_text=vllm_manifest)
run_cmd(["oc", "rollout", "status", f"deployment/{VLLM_NAME}", "-n", NAMESPACE, "--timeout=1800s"]) 

vllm_pod = run_cmd(
    ["oc", "get", "pod", "-n", NAMESPACE, "-l", f"app={VLLM_NAME}", "-o", "jsonpath={.items[0].metadata.name}"]
).stdout.strip()
wait_for_vllm_http_ready(NAMESPACE, vllm_pod, timeout_s=1200)

run_cmd(["oc", "apply", "-f", "-"], input_text=trainjob_manifest)

trainer_pod = ""
for _ in range(180):
    p = run_cmd(
        [
            "oc", "get", "pod", "-n", NAMESPACE,
            "-l", f"jobset.sigs.k8s.io/jobset-name={TRAINJOB_NAME}",
            "-o", "jsonpath={.items[0].metadata.name}",
        ],
        check=False,
    )
    trainer_pod = (p.stdout or "").strip()
    if trainer_pod:
        break
    time.sleep(5)

if not trainer_pod:
    raise RuntimeError("Trainer pod was not created in time")

terminal_state = ""
for _ in range(720):
    p = run_cmd(
        [
            "oc", "get", "trainjob", TRAINJOB_NAME, "-n", NAMESPACE,
            "-o", "jsonpath={.status.conditions[0].type}",
        ],
        check=False,
    )
    terminal_state = (p.stdout or "").strip()
    if terminal_state in {"Complete", "Failed"}:
        print(f"TrainJob terminal condition: {terminal_state}")
        break
    time.sleep(5)

if terminal_state not in {"Complete", "Failed"}:
    raise RuntimeError("TrainJob did not reach terminal condition within timeout")

vllm_node = run_cmd(["oc", "get", "pod", vllm_pod, "-n", NAMESPACE, "-o", "jsonpath={.spec.nodeName}"]).stdout.strip()
trainer_node = run_cmd(["oc", "get", "pod", trainer_pod, "-n", NAMESPACE, "-o", "jsonpath={.spec.nodeName}"]).stdout.strip()

trainer_logs = run_cmd(["oc", "logs", "-n", NAMESPACE, trainer_pod]).stdout
vllm_logs = run_cmd(["oc", "logs", "-n", NAMESPACE, vllm_pod]).stdout

marker = "__CROSS_NODE_RESULT_JSON__"
result_payload = None
for line in trainer_logs.splitlines():
    if line.startswith(marker):
        result_payload = json.loads(line[len(marker):])
        break

if result_payload is None:
    raise RuntimeError("Result marker not found in trainer logs")

cross_node_ok = vllm_node != trainer_node
if TEST_MODE == "cross_node" and not cross_node_ok:
    raise RuntimeError("cross_node mode validation failed: serving and trainer landed on same node")

lifecycle_checks = {
    "start_weight_update_200": 'POST /start_weight_update HTTP/1.1" 200' in vllm_logs,
    "update_weights_200": 'POST /update_weights HTTP/1.1" 200' in vllm_logs,
    "finish_weight_update_200": 'POST /finish_weight_update HTTP/1.1" 200' in vllm_logs,
    "resume_200": 'POST /resume HTTP/1.1" 200' in vllm_logs,
}

print("\nCross-node check:")
print("  vLLM pod:", vllm_pod, "node=", vllm_node)
print("  trainer pod:", trainer_pod, "node=", trainer_node)
print("  cross_node_ok:", cross_node_ok)

summary = {
    "trainjob": TRAINJOB_NAME,
    "trainjob_terminal": terminal_state,
    "test_mode": TEST_MODE,
    "vllm_gpu_count": VLLM_GPU_COUNT,
    "vllm_service": f"http://{VLLM_NAME}.{NAMESPACE}.svc.cluster.local:8000",
    "vllm_pod": vllm_pod,
    "vllm_node": vllm_node,
    "trainer_pod": trainer_pod,
    "trainer_node": trainer_node,
    "cross_node_ok": cross_node_ok,
    "lifecycle_checks": lifecycle_checks,
    "result": result_payload,
}

print("\n__CROSS_NODE_NOTEBOOK_SUMMARY__")
print(json.dumps(summary, indent=2))

if terminal_state != "Complete":
    raise RuntimeError(f"TrainJob ended in non-complete state: {terminal_state}")
if result_payload.get("sync_status") != "success":
    raise RuntimeError(f"sync_status is not success: {result_payload.get('sync_status')} error={result_payload.get('sync_error')}")
if not all(lifecycle_checks.values()):
    raise RuntimeError(f"vLLM lifecycle endpoint checks failed: {lifecycle_checks}")